# Real-Time Fraud Risk Scoring

Dataset: [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

## Problem Statement
Detect fraudulent transactions with high recall and manageable false positives.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

palette_teal_coral = ['#2a9d8f', '#e76f51']
palette_purple_amber = ['#6a4c93', '#ffb703']


## Data Loading and Inspection


In [ ]:
# Replace with actual local file path for this week's dataset
df = pd.read_csv('../data/dataset.csv') if False else pd.DataFrame()
print('Shape:', df.shape)
df.head()


## Exploratory Data Analysis (5+ Visualizations)


In [ ]:
# Viz 1: Missingness heatmap
plt.figure(figsize=(10,4))
sns.heatmap(df.isna(), cbar=False, cmap=sns.color_palette(palette_teal_coral, as_cmap=True))
plt.title('Missingness Map')
plt.xlabel('Features'); plt.ylabel('Rows')
plt.annotate('Data quality overview', xy=(0.02,0.95), xycoords='axes fraction')
plt.show()

# Viz 2: Target distribution
if 'target' in df.columns:
    ax = df['target'].value_counts().plot(kind='bar', color=palette_teal_coral)
    ax.set_title('Target Class Distribution'); ax.set_xlabel('Class'); ax.set_ylabel('Count')
    for p in ax.patches: ax.annotate(int(p.get_height()), (p.get_x()+0.05, p.get_height()+1))
    plt.show()

# Viz 3: Numeric correlation
num_df = df.select_dtypes(include='number')
if not num_df.empty:
    plt.figure(figsize=(8,6))
    sns.heatmap(num_df.corr(), cmap=sns.color_palette(palette_purple_amber, as_cmap=True), annot=False)
    plt.title('Numeric Correlation Heatmap')
    plt.xlabel('Features'); plt.ylabel('Features')
    plt.show()

# Viz 4: Top category frequency
cat_cols = df.select_dtypes(exclude='number').columns
if len(cat_cols) > 0:
    c = cat_cols[0]
    ax = df[c].value_counts().head(10).plot(kind='barh', color=palette_purple_amber[0])
    ax.set_title(f'Top Categories: {c}')
    ax.set_xlabel('Count'); ax.set_ylabel(c)
    plt.show()

# Viz 5: Numeric distribution
if len(num_df.columns) > 0:
    n = num_df.columns[0]
    plt.figure(figsize=(8,4))
    sns.histplot(df[n], color=palette_teal_coral[0], kde=True)
    plt.title(f'Distribution of {n}')
    plt.xlabel(n); plt.ylabel('Frequency')
    plt.annotate('Check skewness/outliers', xy=(0.55,0.9), xycoords='axes fraction')
    plt.show()


## Preprocessing and Feature Engineering


In [ ]:
target_col = 'target'
X = df.drop(columns=[target_col])
y = df[target_col]
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
numeric = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num', numeric, num_cols), ('cat', categorical, cat_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


## Model Training and Comparison (2 Models)


In [ ]:
models = {
    'logreg': LogisticRegression(max_iter=1000),
    'rf': RandomForestClassifier(n_estimators=300, random_state=42)
}
results = {}
for name, mdl in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', mdl)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:,1] if hasattr(pipe, 'predict_proba') else None
    auc = roc_auc_score(y_test, prob) if prob is not None else None
    results[name] = {'report': classification_report(y_test, pred), 'roc_auc': auc}
results


## Results and Key Insights
- Compare model performance metrics and discuss trade-offs.
- Highlight business impacts and operational use-cases.


## Conclusion
Summarize what was learned, what can be deployed, and next improvements.
